# 01 · Building the gene sets

**Project:** Macrocephaly gene landscape  
**Goal of this notebook:** collect the genes linked to macrocephaly, plus three comparison phenotypes (microcephaly, autism, seizures), from the **Human Phenotype Ontology (HPO)**.

**Why HPO?** Its gene–phenotype annotations are openly downloadable, so anyone can rerun this project from scratch.

**Outputs** (used by the next notebooks):
- `results/gene_sets/<phenotype>.txt` – one gene symbol per line
- `results/gene_sets/background_genes.txt` – every gene with at least one HPO annotation (the "universe" for statistics later)
- `results/gene_set_summary.csv`

## 1. Setup
`obonet` reads the ontology file into a network, so we can find every sub-term of a phenotype.

In [ ]:
!pip install -q obonet

import os
import pandas as pd
import networkx as nx
import obonet

os.makedirs("data", exist_ok=True)
os.makedirs("results/gene_sets", exist_ok=True)

## 2. Download the HPO files
- `hp.obo` – the ontology itself (terms and how they relate)
- `genes_to_phenotype.txt` – which genes are annotated to which terms

The download date is recorded, because HPO is updated regularly and gene counts will change slightly between releases.

In [ ]:
from datetime import date

HPO_OBO_URL = "http://purl.obolibrary.org/obo/hp.obo"
G2P_URL = "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/genes_to_phenotype.txt"

!wget -q -O data/hp.obo "{HPO_OBO_URL}"
!wget -q -O data/genes_to_phenotype.txt "{G2P_URL}"

DOWNLOAD_DATE = date.today().isoformat()
print("Downloaded on:", DOWNLOAD_DATE)
!ls -lh data/

## 3. Load the ontology
In `hp.obo`, each edge points from a **child** term to its **parent** (child *is_a* parent).  
So all the more specific sub-terms of a phenotype are its **ancestors in the network graph**. This trips people up, so it is worth remembering.

In [ ]:
hpo = obonet.read_obo("data/hp.obo")
print(f"HPO terms loaded: {len(hpo):,}")

def term_name(hpo_id):
    return hpo.nodes[hpo_id].get("name", "?")

def term_and_subterms(hpo_id):
    """The term itself plus every more specific term below it."""
    return {hpo_id} | nx.ancestors(hpo, hpo_id)

## 4. Choose the phenotypes
**Check the printed names below match what you expect.** If an ID prints the wrong name, fix it here before going further.

In [ ]:
PHENOTYPES = {
    "macrocephaly": "HP:0000256",
    "microcephaly": "HP:0000252",
    "autism":       "HP:0000717",
    "seizure":      "HP:0001250",
}

for label, hpo_id in PHENOTYPES.items():
    n_sub = len(term_and_subterms(hpo_id)) - 1
    print(f"{label:<13} {hpo_id}  ->  {term_name(hpo_id)}  ({n_sub} sub-terms)")

## 5. Load the gene annotations

In [ ]:
g2p = pd.read_csv("data/genes_to_phenotype.txt", sep="\t", comment="#", dtype=str)
print(g2p.shape)
print(g2p.columns.tolist())
g2p.head()

The column names have changed between HPO releases, so this cell finds the gene-symbol and HPO-ID columns whatever they are called.

In [ ]:
def find_col(df, options):
    for c in df.columns:
        if c.strip().lower() in options:
            return c
    raise KeyError(f"None of {options} found in {df.columns.tolist()}")

GENE_COL = find_col(g2p, {"gene_symbol", "entrez-gene-symbol"})
HPO_COL  = find_col(g2p, {"hpo_id", "hpo-term-id"})
print("Gene column:", GENE_COL, "| HPO column:", HPO_COL)

g2p = g2p[[GENE_COL, HPO_COL]].dropna().drop_duplicates()
g2p.columns = ["gene", "hpo_id"]
print(f"Unique gene–term pairs: {len(g2p):,}")

## 6. Build the gene sets
A gene belongs to a phenotype's set if it is annotated to that term **or any of its sub-terms**.  
For example, a gene annotated only to a more specific macrocephaly term still counts as a macrocephaly gene.

In [ ]:
gene_sets = {}
for label, hpo_id in PHENOTYPES.items():
    terms = term_and_subterms(hpo_id)
    genes = set(g2p.loc[g2p["hpo_id"].isin(terms), "gene"])
    gene_sets[label] = genes

background = set(g2p["gene"])
print(f"Background (all HPO-annotated genes): {len(background):,}\n")
for label, genes in gene_sets.items():
    print(f"{label:<13} {len(genes):>5} genes")

## 7. Sanity check
Genes we already expect from the literature (and from the original UG analysis) should be in the macrocephaly set. If most of these are missing, something went wrong above.

In [ ]:
expected = ["PTEN", "MTOR", "PIK3CA", "PIK3CD", "FGFR3", "HRAS", "MECP2", "LRP5", "NSD1", "AKT3"]
macro = gene_sets["macrocephaly"]
for g in expected:
    print(f"{g:<8} {'✓ present' if g in macro else '✗ missing'}")

## 8. Save

In [ ]:
for label, genes in gene_sets.items():
    with open(f"results/gene_sets/{label}.txt", "w") as f:
        f.write("\n".join(sorted(genes)) + "\n")

with open("results/gene_sets/background_genes.txt", "w") as f:
    f.write("\n".join(sorted(background)) + "\n")

summary = pd.DataFrame([
    {"phenotype": label, "hpo_id": PHENOTYPES[label], "hpo_name": term_name(PHENOTYPES[label]),
     "n_genes": len(genes), "hpo_download_date": DOWNLOAD_DATE}
    for label, genes in gene_sets.items()
])
summary.to_csv("results/gene_set_summary.csv", index=False)
summary

## 9. Download the results
Run this, then upload the `results/` folder to the GitHub repository.

In [ ]:
!zip -qr results_01.zip results/
from google.colab import files
files.download("results_01.zip")